# 神经风格迁移实战

本笔记整合原 `neural-style.ipynb`,演示如何使用 PyTorch 将一张内容图像与一张风格图像融合,生成具有目标风格的新图像。示例基于 VGG19 特征网络,已去除对 `d2l` 包的依赖。

## 1. 算法回顾

- **内容损失**: 生成图像与内容图像在高层特征上的差异。
- **风格损失**: 生成图像与风格图像的 Gram 矩阵差异。
- **全变分损失**(可选): 让结果更加平滑。

总体目标:
$$ \mathcal{L} = lpha \cdot \mathcal{L}_{	ext{content}} + eta \cdot \mathcal{L}_{	ext{style}} + \gamma \cdot \mathcal{L}_{	ext{tv}} $$

本笔记默认使用 $lpha=1$, $eta=1e6$, $\gamma=1e-6$,可根据效果调整。

## 2. 环境准备

- 需要 `torch`, `torchvision`, `PIL`, `matplotlib`。
- 由于计算量较大,建议在 GPU 环境运行。



In [ ]:
import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', DEVICE)



## 3. 读取并预处理图像

请准备两张图片(推荐尺寸 < 512×512,避免显存不足),设置下方 `content_path` 与 `style_path`。若图像较大,可在预处理时自动缩放。



In [ ]:
def load_image(path, max_size=512):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'未找到图像: {path}')
    image = Image.open(path).convert('RGB')
    scale = max_size / max(image.size)
    size = tuple(int(dim * scale) for dim in image.size)
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform(image).unsqueeze(0)

content_path = 'path/to/your_content.jpg'
style_path = 'path/to/your_style.jpg'

content_img = load_image(content_path).to(DEVICE)
style_img = load_image(style_path, max_size=content_img.shape[-1]).to(DEVICE)
print('内容图像形状:', content_img.shape)
print('风格图像形状:', style_img.shape)



## 4. 特征提取网络

我们使用 ImageNet 预训练的 VGG19,提取中间特征作为内容与风格表示。为了稳定,仅保留需要的层,并冻结参数。



In [ ]:
class VGGFeatures(nn.Module):
    def __init__(self, content_layers, style_layers):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.eval()
        for param in vgg.parameters():
            param.requires_grad = False

        self.content_layers = content_layers
        self.style_layers = style_layers
        self.model = nn.Sequential()
        self.layer_mapping = {}
        layer_idx = 0
        block = 1
        conv = 0
        for layer in vgg:
            if isinstance(layer, nn.Conv2d):
                conv += 1
                name = f'conv{block}_{conv}'
            elif isinstance(layer, nn.ReLU):
                layer = nn.ReLU(inplace=False)
                name = f'relu{block}_{conv}'
            elif isinstance(layer, nn.MaxPool2d):
                name = f'pool{block}'
                block += 1
                conv = 0
            else:
                name = f'layer_{layer_idx}'
            self.model.add_module(name, layer)
            self.layer_mapping[name] = len(self.model) - 1
            layer_idx += 1

    def forward(self, x):
        content_feats = {}
        style_feats = {}
        for name, layer in self.model._modules.items():
            x = layer(x)
            if name in self.content_layers:
                content_feats[name] = x
            if name in self.style_layers:
                style_feats[name] = x
        return content_feats, style_feats

content_layers = ['conv4_2']
style_layers = ['conv1_1', 'conv2_1', 'conv3_1', 'conv4_1', 'conv5_1']
feature_extractor = VGGFeatures(content_layers, style_layers).to(DEVICE)



## 5. Gram 矩阵

风格表示通过每一层特征图的 Gram 矩阵捕获通道间的相关性。



In [ ]:
def gram_matrix(tensor):
    b, c, h, w = tensor.size()
    features = tensor.view(b, c, h * w)
    gram = torch.bmm(features, features.transpose(1, 2))
    return gram / (c * h * w)



## 6. 构建损失函数

- 内容损失: MSE
- 风格损失: Gram 矩阵的 MSE
- 全变分损失: 对生成图像求一阶差分



In [ ]:
def total_variation_loss(img):
    x_diff = img[:, :, :, :-1] - img[:, :, :, 1:]
    y_diff = img[:, :, :-1, :] - img[:, :, 1:, :]
    return (x_diff.abs().mean() + y_diff.abs().mean())

style_targets = {}
content_targets = {}

with torch.no_grad():
    content_targets, style_targets = feature_extractor(content_img)
    style_targets = {k: gram_matrix(v) for k, v in style_targets.items()}

alpha = 1.0
beta = 1e6
gamma = 1e-6

generated = content_img.clone().requires_grad_(True)
optimizer = torch.optim.LBFGS([generated])



## 7. 迭代优化

使用 L-BFGS 优化生成图像,每次迭代重新计算损失并回传梯度。



In [ ]:
num_steps = 300

run = [0]

while run[0] <= num_steps:
    def closure():
        optimizer.zero_grad()
        gen_content, gen_style = feature_extractor(generated)
        content_loss = 0.0
        style_loss = 0.0

        for name in content_layers:
            content_loss += nn.functional.mse_loss(gen_content[name], content_targets[name])

        for name in style_layers:
            g = gram_matrix(gen_style[name])
            s = style_targets[name]
            style_loss += nn.functional.mse_loss(g, s)

        tv_loss = total_variation_loss(generated)
        loss = alpha * content_loss + beta * style_loss + gamma * tv_loss
        loss.backward()

        if run[0] % 50 == 0:
            print(f"步数 {run[0]}: total={loss.item():.2f}, content={content_loss.item():.2f}, style={style_loss.item():.2f}")
        run[0] += 1
        return loss

    optimizer.step(closure)

with torch.no_grad():
    output = generated.clone()



## 8. 结果可视化与保存

将结果反归一化后展示,并保存到本地。



In [ ]:
unloader = transforms.Compose([
    transforms.Normalize(mean=[0., 0., 0.], std=[1/0.229, 1/0.224, 1/0.225]),
    transforms.Normalize(mean=[-0.485, -0.456, -0.406], std=[1., 1., 1.]),
    transforms.Lambda(lambda x: torch.clamp(x, 0, 1)),
])

def tensor_to_pil(tensor):
    image = unloader(tensor.squeeze().cpu())
    return transforms.ToPILImage()(image)

result_img = tensor_to_pil(output)

plt.figure(figsize=(12, 4))
for i, (title, img_tensor) in enumerate([
    ('内容图像', content_img),
    ('风格图像', style_img),
    ('生成图像', output)
]):
    plt.subplot(1, 3, i + 1)
    plt.title(title)
    plt.axis('off')
    plt.imshow(tensor_to_pil(img_tensor))
plt.show()

save_path = Path('results/style_transfer_result.png')
save_path.parent.mkdir(parents=True, exist_ok=True)
result_img.save(save_path)
print('结果已保存到', save_path)



## 9. 参数调优建议

- 调整 `beta`(风格权重) 可以控制风格程度。
- 采样不同的卷积层组合以改变纹理粒度。
- 对于高分辨率图像,可先运行低分辨率版本,再逐步放大(多阶段策略)。

## 10. 练习

1. 尝试使用另一对内容/风格图像,观察结果差异。
2. 将优化器替换为 Adam,比较收敛速度与图像质量。
3. 为内容损失增加多层约束(如同时使用 `conv4_2` 与 `conv5_2`)。

---
完成风格迁移后,可以继续学习 `03_目标检测与语义分割模型.ipynb`,探索更高级的视觉任务。

